---

HOTEL BOOKING CANCELLATION PREDICTION

---

### **1.Introduction**

#### **1.1 Dataset Overview**

- **Source:** [Hotel Booking Demand — Kaggle](https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand)
- **Total observations (rows):** 119,390
- **Total attributes (columns):** 32 *(original)* → **11 features** after selection & engineering
- **Target variable:** `is_canceled` — Binary class *(0 = Not cancelled, 1 = Cancelled)*
- **Time span:** July 2015 → August 2017
- **Overall cancellation rate:** 37.0%
- **Numerical Features:** `lead_time`, `required_car_parking_spaces`, `total_of_special_requests`,
  `prior_cancel_rate` *(engineered)*, `adr_per_person` *(engineered)*
- **Binary Feature:** `is_short_lead` *(engineered)*
- **Categorical Features:** `hotel`, `deposit_type`, `market_segment`,
  `distribution_channel`, `customer_type`

| Variable Name | Description | Feature Type | Example |
| :--- | :--- | :--- | :--- |
| **is_canceled** ⭐ | Whether the booking was cancelled *(Target)* | Binary | `0` / `1` |
| **lead_time** | Number of days between booking date and arrival date | Numerical | `45` (Days) |
| **required_car_parking_spaces** | Number of parking spaces requested by the guest | Numerical | `1` (Spaces) |
| **total_of_special_requests** | Total number of special requests made (e.g. room floor, bed type) | Numerical | `2` (Requests) |
| **prior_cancel_rate** | Guest's historical cancellation rate, computed with Beta smoothing: $(PC+1)/(PC+PN+2)$ | Numerical *(engineered)* | `0.67` |
| **adr_per_person** | Average daily room rate divided by number of guests; ADR capped at 99.5th percentile | Numerical *(engineered)* | `58.5` (€/person) |
| **is_short_lead** | Binary flag: `1` if booking was made ≤ 7 days before arrival, `0` otherwise | Binary *(engineered)* | `0` / `1` |
| **hotel** | Type of hotel property | Categorical | `Resort Hotel` / `City Hotel` |
| **deposit_type** | Financial commitment type at booking time | Categorical | `No Deposit` / `Non Refund` / `Refundable` |
| **market_segment** | Market segment the booking originated from | Categorical | `Online TA` / `Direct` / `Groups` |
| **distribution_channel** | Channel through which the booking was distributed | Categorical | `TA/TO` / `Direct` / `Corporate` |
| **customer_type** | Classification of the booking customer | Categorical | `Transient` / `Contract` / `Group` |

### **2. Setup**

#### 2.1. Library

In [71]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "optuna", "-q"],
               capture_output=True)

import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, log_loss, brier_score_loss,
)
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings("ignore")
sns.set_theme(context="talk", style="whitegrid", font_scale=0.85)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

print("✅ All libraries loaded.")

✅ All libraries loaded.


#### 2.2. Load data

In [72]:
DATA_PATH = r"..\data\raw\raw_data.csv"

print(f"Loading: {DATA_PATH}")

df_raw = pd.read_csv(DATA_PATH, dtype={"agent": "Float64", "company": "Float64"})

print(f"Shape : {df_raw.shape}   ({df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns)")
print(f"\nTarget (is_canceled) distribution:")
vc = df_raw["is_canceled"].value_counts()
print(f"  Not cancelled (0) : {vc[0]:,}  ({vc[0]/len(df_raw):.1%})")
print(f"  Cancelled     (1) : {vc[1]:,}  ({vc[1]/len(df_raw):.1%})")



Loading: ..\data\raw\raw_data.csv
Shape : (119390, 32)   (119,390 rows, 32 columns)

Target (is_canceled) distribution:
  Not cancelled (0) : 75,166  (63.0%)
  Cancelled     (1) : 44,224  (37.0%)


### 3. Data Audit
Before touching anything, we check:
- pct_null   – columns with high missingness need special handling
- n_unique   – very high cardinality (e.g. 177 countries) is expensive to encode
- dtype      – confirm numeric columns are not stored as strings


In [73]:
print("=== dtype / missing-value audit ===")
audit = pd.DataFrame({
    "dtype"    : df_raw.dtypes,
    "n_null"   : df_raw.isna().sum(),
    "pct_null" : (df_raw.isna().mean() * 100).round(1),
    "n_unique" : df_raw.nunique(),
})
print(audit.sort_values("pct_null", ascending=False).to_string())



=== dtype / missing-value audit ===
                                  dtype  n_null  pct_null  n_unique
company                         Float64  112593      94.3       352
agent                           Float64   16340      13.7       333
country                             str     488       0.4       177
hotel                               str       0       0.0         2
arrival_date_month                  str       0       0.0        12
arrival_date_week_number          int64       0       0.0        53
lead_time                         int64       0       0.0       479
is_canceled                       int64       0       0.0         2
stays_in_weekend_nights           int64       0       0.0        17
stays_in_week_nights              int64       0       0.0        35
children                        float64       4       0.0         5
adults                            int64       0       0.0        14
babies                            int64       0       0.0         5
meal        

### 4. LEAKAGE AUDIT
Identify data leakage features to prevent the model from learning information unavailable at prediction time
| Feature | Status | Reason |
|---|---|---|
| `reservation_status` | ❌ LEAKAGE | Directly encodes the outcome. |
| `reservation_status_date` | ❌ LEAKAGE | Only exists for cancelled rows. |
| `country` | ⚠️ EXCLUDED | 177 categories + partial leakage risk. |
| `agent` | ⚠️ EXCLUDED | 333 unique IDs; high cardinality. |
| `company` | ⚠️ EXCLUDED | 94% missing. |
| `arrival_date_*` | 📅 KEPT (meta) | Not a feature; used only to split by time. |

### 5. SELECT SOURCE COLUMNS
Select raw columns that feed into our final feature table.

Three extra date columns are added purely so we can reconstruct arrival_date later for the time-based split — they are dropped before model training.
| Column | Type | Purpose |
|---|---|---|
| `lead_time` | Numerical | Number of days between booking and arrival. |
| `required_car_parking_spaces` | Numerical | Availability of car parking spaces. |
| `total_of_special_requests` | Numerical | Total number of special customer requests. |
| `previous_cancellations` | Numerical | Number of previous cancelled bookings. |
| `previous_bookings_not_canceled` | Numerical | Number of previous successful bookings. |
| `adr` | Numerical | Average Daily Rate of the booking. |
| `adults` | Numerical | Number of adults in the booking. |
| `children` | Numerical | Number of children in the booking. |
| `babies` | Numerical | Number of babies in the booking. |
| `deposit_type` | Categorical | Deposit/payment type. |
| `market_segment` | Categorical | Booking market segment. |
| `distribution_channel` | Categorical | Booking distribution channel. |
| `customer_type` | Categorical | Type of customer. |
| `hotel` | Categorical | Hotel type. |
| `arrival_date_year` | Date Metadata | Used for time-based splitting. |
| `arrival_date_month` | Date Metadata | Used for time-based splitting. |
| `arrival_date_day_of_month` | Date Metadata | Used for time-based splitting. |
| `is_canceled` | Target | Booking cancellation outcome. |


In [74]:

TARGET = "is_canceled"

DATE_COLS   = ["arrival_date_year", "arrival_date_month", "arrival_date_day_of_month"]
SOURCE_COLS = [
    # raw numerics → used directly or as ingredients for engineered features
    "lead_time", "required_car_parking_spaces", "total_of_special_requests",
    "previous_cancellations", "previous_bookings_not_canceled",
    "adr", "adults", "children", "babies",
    # categoricals → used directly after one-hot encoding
    "deposit_type", "market_segment", "distribution_channel", "customer_type", "hotel",
] + DATE_COLS

df = df_raw[SOURCE_COLS + [TARGET]].copy()
df["children"] = df["children"].fillna(0).astype(int)  # fill 4 missing children values

print(f"Working frame: {df.shape}")
print(f"Columns: {df.columns.tolist()}")



Working frame: (119390, 18)
Columns: ['lead_time', 'required_car_parking_spaces', 'total_of_special_requests', 'previous_cancellations', 'previous_bookings_not_canceled', 'adr', 'adults', 'children', 'babies', 'deposit_type', 'market_segment', 'distribution_channel', 'customer_type', 'hotel', 'arrival_date_year', 'arrival_date_month', 'arrival_date_day_of_month', 'is_canceled']


In [75]:
### 6. LOGIC CHECKING

In [76]:
# 1. Negative values check
negative_cols = [
    "lead_time",
    "adr",
    "adults",
    "children",
    "babies",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "total_of_special_requests"
]

for col in negative_cols:
    n_neg = (df[col] < 0).sum()
    print(f"{col:<35} negative values: {n_neg}")

# 2. Guest count consistency
df["total_guests"] = df["adults"] + df["children"] + df["babies"]

zero_guest_rows = (df["total_guests"] == 0).sum()
print(f"\nRows with zero guests: {zero_guest_rows}")

# 3. Missing values check
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0]

if len(missing_values) == 0:
    print("\nNo missing values remaining.")
else:
    print(missing_values.sort_values(ascending=False))

# 4. Duplicate rows
duplicates = df.duplicated().sum()
print(f"\nDuplicate rows: {duplicates}")

lead_time                           negative values: 0
adr                                 negative values: 1
adults                              negative values: 0
children                            negative values: 0
babies                              negative values: 0
previous_cancellations              negative values: 0
previous_bookings_not_canceled      negative values: 0
total_of_special_requests           negative values: 0

Rows with zero guests: 180

No missing values remaining.

Duplicate rows: 36852


In [77]:
print("=" * 55)
print("DATA QUALITY CHECK")
print("=" * 55)

# FIX 1 — Negative values
before = len(df)
df = df[df["adr"] >= 0].copy()
print(f"\n  → FIX: Dropped {before - len(df)} row(s) with negative ADR.")
print(f"         Remaining: {len(df):,} rows")

# FIX 2 — Zero-guest rows
before = len(df)
df = df[df["total_guests"] > 0].copy()
df = df.drop(columns=["total_guests"])
print(f"\n  → FIX: Dropped {before - len(df)} zero-guest row(s).")
print(f"         Remaining: {len(df):,} rows")

# FIX 3 — Duplicate rows
# Keep the first occurrence
before = len(df)
df = df.drop_duplicates(keep="first").reset_index(drop=True)
print(f"\n  → FIX: Removed {before - len(df):,} duplicate row(s). Kept first occurrence.")
print(f"         Remaining: {len(df):,} rows")

DATA QUALITY CHECK

  → FIX: Dropped 1 row(s) with negative ADR.
         Remaining: 119,389 rows

  → FIX: Dropped 180 zero-guest row(s).
         Remaining: 119,209 rows

  → FIX: Removed 36,832 duplicate row(s). Kept first occurrence.
         Remaining: 82,377 rows


### 5. Save data

In [78]:
df.to_csv("..\data\cleaned\cleaned_data.csv", index=False)